# Module 3 · Variant calling
**Xenoi Biotech · DNA-Seq (Variant Analysis) Workshop** · Week 2 · Days 11–15

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjalianjali1234/dna-seq-variant-colab/blob/main/notebooks/M3_Variant_calling.ipynb)

**By the end of this module:** you can call SNPs and indels with GATK HaplotypeCaller, BCFtools and FreeBayes, understand single- vs multi-sample calling, apply hard filters and compare callers against a truth set.

**Workshop question:** can different variant callers recover the same small DNA changes in a real human sample (HG001 / NA12878), and how do we know which calls to trust?

**How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code under it. Cells marked ✍️ are exercises: write your answers in your lab notebook.

## Session setup (run every time you open Colab)
Colab gives you a fresh computer each session, so this cell installs the tools and downloads the data. The first run takes about **3–5 minutes**; go through the reading below while it works.

In [ ]:
import os
%cd /content
if not os.path.exists("dna-seq-variant-colab"):
    !git clone -q https://github.com/anjalianjali1234/dna-seq-variant-colab.git
%cd /content/dna-seq-variant-colab
!bash setup/colab_setup.sh
os.environ["PATH"] = "/content/env/bin:" + os.environ["PATH"]
!samtools --version | head -1 && gatk --version 2>/dev/null | head -1

### Catch up on earlier modules
This rebuilds the outputs of Modules 1–2 (about 2–4 minutes). Skip it if you ran them in this same session.

In [ ]:
!bash setup/catchup.sh 2

> 💾 **Optional:** to keep your outputs between sessions, mount Google Drive and copy `run/` there at the end:
> `from google.colab import drive; drive.mount('/content/drive')` then `!cp -r run /content/drive/MyDrive/dna-seq-run`

## Day 11 · SNPs, indels and how callers think
- **SNP/SNV**: one base changed (C>T). **Indel**: bases inserted or deleted.
- **GATK HaplotypeCaller** re-assembles reads into local haplotypes: strong for indels.
- **BCFtools** computes genotype likelihoods from the pileup at each position: fast and simple.
- **FreeBayes** is a Bayesian, haplotype-based caller.

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
bash scripts/call_variants.sh
for c in gatk bcftools freebayes; do
  printf "%-10s %s variants\n" $c $(bcftools view -H run/calls/$c.norm.vcf.gz | wc -l)
done

> ✍️ **Your turn**
>
> Why do we **normalise** (left-align and split multi-allelic sites) before comparing callers? Give an example of one indel written two different ways.

## Day 12 · Read a genotype

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t%QUAL[\t%GT\t%DP\t%AD]\n' run/calls/gatk.norm.vcf.gz | head -8

> ✍️ **Your turn**
>
> 1. For one heterozygous SNV, compute the allele balance = ALT reads / (REF + ALT reads).
> 2. Explain what GT, QUAL and DP each tell you.
> 3. Why might a true heterozygote show 35% instead of 50% ALT reads?

## Day 13 · Calling strategies: single-sample and multi-sample
For a cohort, GATK uses the **gVCF** workflow: each sample is called with `-ERC GVCF`, which also records confidence at **non-variant** positions; samples are then combined and genotyped together. Here we run it for our one sample so you see the files.

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
gatk HaplotypeCaller -R run/input/ref.fa -I run/align/HG001.bam -L chr20:10000001-10010000 \
    -ERC GVCF -O run/calls/HG001.g.vcf.gz --verbosity ERROR 2>/dev/null
gatk GenotypeGVCFs -R run/input/ref.fa -V run/calls/HG001.g.vcf.gz -O run/calls/HG001.joint.vcf.gz --verbosity ERROR 2>/dev/null
bcftools view -H run/calls/HG001.g.vcf.gz | head -3
echo "variant records after joint genotyping: $(bcftools view -H run/calls/HG001.joint.vcf.gz | wc -l)"

> ✍️ **Your turn**
>
> 1. What does a `<NON_REF>` block in the gVCF mean?
> 2. Why is merging separately-called variant-only VCFs risky in a cohort? (Hint: absent call ≠ confident reference.)
> 3. For 3 samples you would run `GenomicsDBImport` or `CombineGVCFs` before `GenotypeGVCFs`. Sketch the command order.

## Day 14 · Hard filters and quality scores

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
for q in 20 30 60; do
  for dp in 10 20 40; do
    printf "QUAL>=%-3s DP>=%-3s  GATK=%s\n" $q $dp \
      $(bcftools view -H -i "QUAL>=$q && FORMAT/DP>=$dp" run/calls/gatk.norm.vcf.gz | wc -l)
  done
done

GATK also has recommended **hard filters** using annotations such as QD (quality by depth), FS (strand bias) and MQ (mapping quality). The full ML-based alternative, VQSR, needs thousands of variants, so it is not used on a small region.

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
gatk VariantFiltration -R run/input/ref.fa -V run/calls/gatk.raw.vcf.gz -O run/calls/gatk.hardfilt.vcf.gz \
  --filter-expression "QD < 2.0" --filter-name QD2 --filter-expression "FS > 60.0" --filter-name FS60 \
  --filter-expression "MQ < 40.0" --filter-name MQ40 --verbosity ERROR 2>/dev/null
bcftools query -f '%FILTER\n' run/calls/gatk.hardfilt.vcf.gz | sort | uniq -c

## Day 15 · Comparing callers against the truth set
`benchmark.py` keeps only alleles inside the confident regions of the NIST truth set, then counts true positives (TP), false positives (FP) and false negatives (FN).
**Precision** = TP / (TP + FP) · **Recall** = TP / (TP + FN)

In [ ]:
!python scripts/benchmark.py 30 10

In [ ]:
import pandas as pd
a = pd.read_csv("run/report/alleles_q30_dp10.tsv", sep="\t")
inside = a[a.benchmark != "outside"]
print("Agreement pattern (gatk, bcftools, freebayes):")
print(inside.groupby(["gatk", "bcftools", "freebayes", "benchmark"]).size())

> ✍️ **Your turn**
>
> **Module 3 deliverable:** a table of TP, FP, FN, precision and recall for the three callers, plus 3 sentences: which caller performed best here, where they disagreed, and why one 10 kb region cannot rank callers in general.